# 🧠 Model Training & Data Preparation (YOLOv8)

Este cuaderno abarca la primera fase del proyecto: la preparación de los datos, el entrenamiento del modelo de detección de defectos y la simulación del entorno de producción.

**Objetivos de esta fase:**
1. Descargar y preparar el dataset industrial **NEU-DET** (Steel Surface Defect Database).
2. Entrenar un modelo **YOLOv8** optimizado para identificar 6 tipos de defectos en el acero laminado.
3. Generar un flujo de vídeo sintético a partir de imágenes estáticas para simular las condiciones físicas de una cinta transportadora en la fábrica.

## 1. Configuración del Entorno y Dataset (NEU-DET)
Instalamos las librerías necesarias y nos conectamos a Roboflow para descargar el dataset estandarizado. El dataset NEU-DET contiene imágenes de la superficie del acero con 6 clases de defectos críticos: *crazing, inclusion, patches, pitted_surface, rolled-in_scale* y *scratches*.

In [3]:
!pip install ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2


In [4]:
from roboflow import Roboflow
rf = Roboflow(api_key="jMu7svX1hJ59eRDGIRzN")
project = rf.workspace("imperial-college-tcys7").project("neu-det-hsegd")
version = project.version(2)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to NEU-DET-2 in yolov8:: 100%|██████████| 3610/3610 [00:01<00:00, 2291.41it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


## 2. Entrenamiento del Modelo de Visión (YOLOv8n)
Inicializamos el modelo YOLOv8 en su versión 'nano' (la más ligera y rápida, ideal para entornos industriales con recursos limitados) y lanzamos el entrenamiento. Usamos `imgsz=640` para estandarizar la resolución de entrada.

In [ ]:
from ultralytics import YOLO

# 1. Cargar un modelo preentrenado (usamos la versión 'nano', que es la más rápida)
model = YOLO('yolov8n.pt')

# 2. Entrenar el modelo
# Ajustamos a 20 épocas para ver resultados rápidos.
results = model.train(data='/content/NEU-DET-2/data.yaml', epochs=100, imgsz=640)

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/NEU-DET-2/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=None, optimize

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/runs/detect/train-2 "/content/drive/MyDrive/Computer vision/"

## 3. Generación de Datos Sintéticos (Simulador de Producción)
Para validar el modelo en condiciones similares a las de una fábrica real, necesitamos procesar un flujo de vídeo. Dado que el dataset NEU-DET solo contiene imágenes estáticas, hemos desarrollado un script que las concatena para simular una cinta transportadora de acero laminado.

**Consideración para Object Tracking:** Para que algoritmos de seguimiento modernos (como BoT-SORT) asignen un ID único a un defecto, necesitan coherencia temporal. Por ello, cada imagen se repite secuencialmente durante 15 fotogramas.

In [9]:
import cv2
import glob

In [10]:
ruta_imagenes = '/content/NEU-DET-2/train/images/*.jpg'
lista_imagenes = glob.glob(ruta_imagenes)

In [13]:
if len(lista_imagenes) > 0:
    img_ejemplo = cv2.imread(lista_imagenes[0])
    alto, ancho, _ = img_ejemplo.shape

    ruta_video_salida = '/content/drive/MyDrive/Computer vision/train_piezas_defectuosas/cinta_acero.mp4'

    video_sintetico = cv2.VideoWriter(ruta_video_salida, cv2.VideoWriter_fourcc(*"mp4v"), 2.0, (ancho, alto))

    for ruta in lista_imagenes[:6]:
        img = cv2.imread(ruta)
        # Escribimos el mismo frame un par de veces para ayudar al Tracker
        for _ in range(15):
          video_sintetico.write(img)

    video_sintetico.release()
    print(f"🎬 Vídeo simulado creado y guardado en: {ruta_video_salida}")
else:
    print("❌ No se encontraron imágenes en la ruta.")

🎬 Vídeo simulado creado y guardado en: /content/drive/MyDrive/Computer vision/train_piezas_defectuosas/cinta_acero.mp4
